In [15]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.distributions import Categorical
import numpy as np
from CharRNN import RNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
import re
import torch

CHECKPOINT_PATH = "./preTrained/CharRNN_shakespeare_Chenglong.pth"


def clean_key(name):
    if name.startswith("module."):
        return name[len("module."):]
    return name


def infer_recurrent_type(weight_hh):
    rows, cols = weight_hh.shape

    if rows == cols:
        return "RNN", 1
    if rows == 3 * cols:
        return "GRU", 3
    if rows == 4 * cols:
        return "LSTM", 4

    return "Unknown", rows / cols


def main():
    state = torch.load(CHECKPOINT_PATH, map_location="cpu")

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    state = {clean_key(k): v for k, v in state.items() if torch.is_tensor(v)}

    print(f"Checkpoint: {CHECKPOINT_PATH}")
    print("=" * 80)

    print("\nAll tensors:")
    for name, tensor in state.items():
        print(f"{name:30s} shape={tuple(tensor.shape)} params={tensor.numel():,}")

    print("\nModel structure:")
    print("-" * 80)

    if "embedding.weight" in state:
        emb = state["embedding.weight"]
        print(f"Embedding:")
        print(f"  vocab size      : {emb.shape[0]}")
        print(f"  embedding dim   : {emb.shape[1]}")
        print(f"  shape           : {tuple(emb.shape)}")

    recurrent_layers = []

    for name in state:
        m = re.match(r"rnn\.weight_hh_l(\d+)$", name)
        if m:
            recurrent_layers.append(int(m.group(1)))

    recurrent_layers = sorted(recurrent_layers)

    if recurrent_layers:
        first_hh = state[f"rnn.weight_hh_l{recurrent_layers[0]}"]
        rnn_type, gate_multiplier = infer_recurrent_type(first_hh)

        print(f"\nRecurrent module:")
        print(f"  inferred type   : {rnn_type}")
        print(f"  num layers      : {len(recurrent_layers)}")
        print(f"  hidden size     : {first_hh.shape[1]}")
        print(f"  gate multiplier : {gate_multiplier}")

        for layer in recurrent_layers:
            print(f"\n  Layer {layer}:")

            for suffix in [
                "weight_ih",
                "weight_hh",
                "bias_ih",
                "bias_hh",
            ]:
                key = f"rnn.{suffix}_l{layer}"
                if key in state:
                    tensor = state[key]
                    print(f"    {suffix:10s} shape={tuple(tensor.shape)} params={tensor.numel():,}")

    if "decoder.weight" in state:
        dec_w = state["decoder.weight"]
        print(f"\nDecoder:")
        print(f"  output vocab    : {dec_w.shape[0]}")
        print(f"  input hidden    : {dec_w.shape[1]}")
        print(f"  weight shape    : {tuple(dec_w.shape)}")

    if "decoder.bias" in state:
        print(f"  bias shape      : {tuple(state['decoder.bias'].shape)}")

    print("\nParameter count by block:")
    print("-" * 80)

    groups = {
        "embedding": 0,
        "rnn": 0,
        "decoder": 0,
        "other": 0,
    }

    for name, tensor in state.items():
        if name.startswith("embedding."):
            groups["embedding"] += tensor.numel()
        elif name.startswith("rnn."):
            groups["rnn"] += tensor.numel()
        elif name.startswith("decoder."):
            groups["decoder"] += tensor.numel()
        else:
            groups["other"] += tensor.numel()

    total = sum(groups.values())

    for group, count in groups.items():
        print(f"{group:10s}: {count:,}")

    print(f"{'total':10s}: {total:,}")


if __name__ == "__main__":
    main()

Checkpoint: ./preTrained/CharRNN_shakespeare_Chenglong.pth

All tensors:
embedding.weight               shape=(66, 66) params=4,356
rnn.weight_ih_l0               shape=(512, 66) params=33,792
rnn.weight_hh_l0               shape=(512, 512) params=262,144
rnn.bias_ih_l0                 shape=(512,) params=512
rnn.bias_hh_l0                 shape=(512,) params=512
rnn.weight_ih_l1               shape=(512, 512) params=262,144
rnn.weight_hh_l1               shape=(512, 512) params=262,144
rnn.bias_ih_l1                 shape=(512,) params=512
rnn.bias_hh_l1                 shape=(512,) params=512
rnn.weight_ih_l2               shape=(512, 512) params=262,144
rnn.weight_hh_l2               shape=(512, 512) params=262,144
rnn.bias_ih_l2                 shape=(512,) params=512
rnn.bias_hh_l2                 shape=(512,) params=512
decoder.weight                 shape=(66, 512) params=33,792
decoder.bias                   shape=(66,) params=66

Model structure:
------------------------------

In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Categorical

DATA_PATH = "./data/shakespeare.txt"
CHECKPOINT_PATH = "./preTrained/CharRNN_shakespeare_Chenglong.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


with open(DATA_PATH, "r") as f:
    raw_data = f.read()

chars = sorted(list(set(raw_data)))
vocab_size = len(chars)

char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

print("vocab_size:", vocab_size)


state = torch.load(CHECKPOINT_PATH, map_location=device)

hidden_size = state["rnn.weight_hh_l0"].shape[1]
num_layers = len([
    k for k in state.keys()
    if k.startswith("rnn.weight_hh_l")
])

print("hidden_size:", hidden_size)
print("num_layers:", num_layers)

device: cpu
vocab_size: 66
hidden_size: 512
num_layers: 3


In [ ]:
def manual_rnn_step(token_id, hidden):
    """
    token_id: int
    hidden: shape = (num_layers, 1, hidden_size)
    """

    # x_t = embedding[token_id]
    x = state["embedding.weight"][token_id].view(1, -1)

    new_hidden_layers = []

    for layer in range(num_layers):
        W_ih = state[f"rnn.weight_ih_l{layer}"]
        W_hh = state[f"rnn.weight_hh_l{layer}"]
        b_ih = state[f"rnn.bias_ih_l{layer}"]
        b_hh = state[f"rnn.bias_hh_l{layer}"]

        h_prev = hidden[layer]

        h_t = torch.tanh(
            x @ W_ih.t() +
            b_ih +
            h_prev @ W_hh.t() +
            b_hh
        )

        new_hidden_layers.append(h_t)


        x = h_t

    new_hidden = torch.stack(new_hidden_layers, dim=0)

    W_dec = state["decoder.weight"]
    b_dec = state["decoder.bias"]

    # logits = h_last W_decoder^T + b_decoder
    logits = x @ W_dec.t() + b_dec

    return logits, new_hidden

In [19]:
def manual_forward(text):

    hidden = torch.zeros(num_layers, 1, hidden_size, device=device)

    logits = None

    for ch in text:
        if ch not in char_to_ix:
            raise ValueError(f"Character not in vocabulary: {repr(ch)}")

        token_id = char_to_ix[ch]
        logits, hidden = manual_rnn_step(token_id, hidden)

    return logits, hidden

In [20]:
seed = "KING "

logits, hidden = manual_forward(seed)
probs = F.softmax(logits[0], dim=0)

top_probs, top_ids = torch.topk(probs, k=10)

for p, idx in zip(top_probs, top_ids):
    print(repr(ix_to_char[idx.item()]), float(p))

'H' 0.15445789694786072
'C' 0.15157951414585114
'E' 0.08907671272754669
'D' 0.0764291062951088
'S' 0.06709181517362595
'J' 0.06291774660348892
'F' 0.05793264880776405
'O' 0.05503396317362785
'G' 0.04672177881002426
'K' 0.041302744299173355


In [24]:
def manual_generate(seed_text="KING ", num_chars=500, temperature=0.8):
    if len(seed_text) == 0:
        raise ValueError("seed_text cannot be empty")

    if temperature <= 0:
        raise ValueError("temperature must be > 0")

    logits, hidden = manual_forward(seed_text)
    generated = seed_text

    for _ in range(num_chars):
        scaled_logits = logits[0] / temperature
        probs = F.softmax(scaled_logits, dim=0)

        dist = Categorical(probs)
        next_id = dist.sample().item()
        next_char = ix_to_char[next_id]

        generated += next_char

        logits, hidden = manual_rnn_step(next_id, hidden)

    return generated


print(manual_generate(seed_text="God", num_chars=1000, temperature=0.5))

God:
Here is the devil that dead and of the princes.

FLUELLEN:
The man of the doors.

KING HENRY V:
What is the best in their and constable and
guide the subjects of the king,
And in the world of this knight:
The king is our part, and the officer, and then go to the
strange the ground and a trumpet;
And there is not gone, the ground to the soldier: but this is the moonship.

KING HENRY V:
What's the dead and priest, and then,
Can thou now the old all the good conscience to the earth of the king;
And the king is as the soldier can be not a good words,
And the weather's tongue in the stone.

KING HENRY V:
I will, my lord, I will be our thoughts of the Good.

KING HENRY V:
The king and the king, that thou hast been ten thousand grands to parced to the Dauphin
to the officers of France.

WILLIAMS:
Your pleasure, and that you will not be the king's throat,
And look you, lords, they shall speak the king?

KING HENRY V:
How now! who came a great knife of the charge,
That knows the devil.

KI